## Importations

In [ ]:
import numpy as np
import pandas as pd

from pandas.api.types import is_numeric_dtype

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro, chi2_contingency, zscore, f_oneway, pearsonr, spearmanr

import plotly.express as px

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_style("whitegrid")

### Database connection

MySQL database with 
- players
- player_season_stats
- teams
- champ
- mvp_votes
- seasons 

tables.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASS")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{db}"
)

In [ ]:
query_players = """
SELECT *
FROM players
"""
query_player_stats = """
SELECT *
FROM player_season_stats
"""
query_teams = """
SELECT *
FROM teams
"""
query_champ = """
SELECT *
FROM champ
"""
query_mvp = """
SELECT *
FROM mvp_votes
"""
query_season = """
SELECT *
FROM seasons
"""

players = pd.read_sql(query_players, engine)
players_stats = pd.read_sql(query_player_stats, engine)
teams = pd.read_sql(query_teams, engine)
champ = pd.read_sql(query_champ, engine)
mvp = pd.read_sql(query_mvp, engine)
seasons = pd.read_sql(query_season, engine)

Build the season-level analysis table: one row per player per season,
joined with player biography, team name, and season label.

Merge correctness notes:
- player_id is a primary key on `players`, so this left join cannot fan out
 player_season_stats rows -- row count after the merge equals row count before.
- `players_stats` and `players` both have a "position" column; suffixes=("", "_bio")
 keeps the LEFT frame's (season-specific) "position" unchanged, which is the one
 `position_main` below correctly uses -- not the player's career-long position.

In [ ]:

data = players_stats.merge(players[["player_id", "name", "height_cm", "weight_kg", "position", "birth_place", "shoots",
                     "draft_year", "nba_debut", "experience_years", "is_active"]],
           on="player_id", how="left", suffixes=("", "_bio")).merge(teams[["team_id", "team_name"]], on="team_id", how="left").merge(seasons, on="season_id", how="left")

# the career-position column created by the "position" name collision above is
# never used (position_main uses the season-specific "position" instead) -- drop it
data = data.drop(columns=["position_bio"])


In [ ]:
# core derived per-game / shooting metrics
data["fg_pct"] = data["field_goals"] / data["field_goals_attempted"]
data["ft_pct"] = data["free_throws"] / data["free_throws_attempted"]
data["points_per_game"] = data["points"] / data["games"]
data["assists_per_game"] = data["assists"] / data["games"]
data["rebounds_per_game"] = (data["offensive_rebounds"] + data["defensive_rebounds"]) / data["games"]
data["true_shooting_pct"] = data["points"] / (2 * (data["field_goals_attempted"] + 0.44 * data["free_throws_attempted"]))
data["position_main"] = data["position"].str.split(" and ").str[0]

 Season-specific experience fix :

 players.experience_years / players.is_active are the player's CURRENT totals, not what they were during this particular season row. Attaching that static value to every historical season is misleading for any season-level test (e.g. "experience_years_bin" would overstate how experienced a player was in an old season if they're still playing today). Reconstruct experience AT THE TIME OF EACH SEASON from nba_debut instead, and keep the static column too but rename it so its "current, not season-specific" meaning is explicit.

In [ ]:
data = data.rename(columns={"experience_years": "experience_years_current_total"})
data["season_year"] = data["season"].astype(str).str[:4].astype(int)
data["experience_years"] = (data["season_year"] - data["nba_debut"]).clip(lower=0)

data.shape
data.shape

#### First look at the data

In [ ]:
data.info()

In [ ]:
data.head(10)

In [ ]:
print("The dataset contains", data.shape[0], "rows and", data.shape[1], "columns.")
print("The dataset contains", data.duplicated().sum(), "duplicate rows.")

In [ ]:
# Target: did this player-season receive at least one MVP vote?
# This is the binary outcome used throughout the "vs target" analysis below,
# playing the same role the Anxious/Not-Anxious label plays in the reference notebook.
mvp_pairs = set(zip(mvp["player_id"], mvp["season_id"]))
data["Target"] = data.apply(lambda r: int((r["player_id"], r["season_id"]) in mvp_pairs), axis=1)
target = "Target"
data["Target"].value_counts()

In [ ]:
numeric_columns = ["minutes_played", "field_goals", "field_goals_attempted",
                   "three_point_field_goals", "three_point_field_goals_attempted",
                   "two_point_field_goals", "two_point_field_goals_attempted",
                   "free_throws", "free_throws_attempted",
                   "offensive_rebounds", "defensive_rebounds", "assists", "steals", "blocks",
                   "turnovers", "points", "personal_fouls", "triple_doubles",
                   "points_per_game", "assists_per_game", "rebounds_per_game",
                   "fg_pct", "ft_pct", "true_shooting_pct", "height_cm", "weight_kg",
                   "games", "games_started", "experience_years"]

categorical_columns = ["position_main", "team_name", "season", "shoots",
                       "is_active", "Target"]

In [ ]:
data[numeric_columns].describe().T

## Duplicated

In [ ]:
def duplicate_indicator(df: pd.DataFrame) -> str:
    dup_num = df.duplicated().sum()
    return f"Dataframe contains {dup_num} duplicated entries"

In [ ]:
duplicate_indicator(data)

## Before Cleaning

### Normality

In [ ]:
def normality_checker(df, columns=None, alpha=0.05) -> pd.DataFrame:

    if columns is None:
        columns = df.select_dtypes(include="number").columns

    results = []

    for col in columns:
        clean_data = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        n = len(clean_data)

        if n < 3:
            results.append([col, n, None, "Not enough data"])
            continue

        sample = clean_data.sample(min(n, 5000), random_state=42)
        stat, p_value = shapiro(sample)

        result = "Normal" if p_value > alpha else "Not normal"

        results.append([col, n, p_value, result])

    result_df = pd.DataFrame(
        results,
        columns=["Column", "Sample Size", "p-value", "Result"]
    )

    return result_df

In [ ]:
normality_checker(data, numeric_columns)

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(20, 20))
axes = axes.flatten()

for ax, c in zip(axes, numeric_columns):
    sns.histplot(data[c].replace([np.inf, -np.inf], np.nan), kde=True, ax=ax)
    ax.axvline(data[c].mean(), color="red", linestyle="--", label="mean")
    ax.axvline(data[c].median(), color="green", linestyle="--", label="median")
    ax.set_title(c, fontsize=13)
    ax.legend(fontsize=10)

for ax in axes[len(numeric_columns):]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()

### NaN percentage

In [ ]:
def nan_percentage(df: pd.DataFrame) -> pd.DataFrame:
    total_rows = len(df)
    missing_count = [df[col].isna().sum() for col in df.columns]
    nan_pct = [(df[col].isna().sum() / total_rows) * 100 for col in df.columns]
    result = pd.DataFrame({"Missing Count": missing_count, "Nan Percent": nan_pct}, index=df.columns)
    return result.sort_values(by="Nan Percent", ascending=False)

In [ ]:
nan_percentage(data).head(10)

In [ ]:
# draft_year is missing precisely for undrafted players -- check the relationship
# between missing draft_year and having a recorded nba_debut
pd.crosstab(data["draft_year"].isna(), data["nba_debut"].notna())

## Cleaning Data

In [ ]:
# draft_year is genuinely missing for undrafted players -- this is meaningful information,
# not noise, so it gets an explicit label instead of being dropped or median-filled
data["draft_year"] = data["draft_year"].astype("Int64")
data["is_undrafted"] = data["draft_year"].isna().map({True: "Undrafted", False: "Drafted"})
data["is_undrafted"].value_counts()

### Impute NaNs

In [ ]:
def fill_numeric(data, cols):
    data = data.copy()
    for c in cols:
        if data[c].isnull().sum() > 0:
            med = data[c].median()
            print(c, "->", data[c].isnull().sum(), "filled with median", round(med, 3))
            data[c] = data[c].fillna(med)
    return data

def fill_categorical(data, cols):
    data = data.copy()
    for c in cols:
        if data[c].isnull().sum() > 0:
            m = data[c].mode()[0]
            print(c, "->", data[c].isnull().sum(), "filled with mode", m)
            data[c] = data[c].fillna(m)
    return data

In [ ]:
numeric_to_fill = ["fg_pct", "ft_pct", "true_shooting_pct", "height_cm", "weight_kg"]
categorical_to_fill = ["shoots", "position_main"]

In [ ]:
data = fill_numeric(data, numeric_to_fill)
data = fill_categorical(data, categorical_to_fill)

In [ ]:
print("Remaining missing values (excluding draft_year, kept as-is):",
      data.drop(columns=["draft_year"]).isnull().sum().sum())

## Outliers, Binning & Reclassifying

### Outliers

In [ ]:
def outlier_detector(col, method="iqr", k=1.5, z_thresh=3, clip=False) -> pd.Series:
    col = col.replace([np.inf, -np.inf], np.nan)

    if method == "iqr":
        q1, q3 = col.quantile(0.25), col.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - k * iqr
        upper = q3 + k * iqr
        mask = (col < lower) | (col > upper)

        if clip:
            return col.clip(lower, upper)
        return mask

    elif method == "zscore":
        z = col.sub(col.mean()).div(col.std()).abs()
        mask = z > z_thresh

        if clip:
            mean, std = col.mean(), col.std()
            lower = mean - z_thresh * std
            upper = mean + z_thresh * std
            return col.clip(lower, upper)
        return mask

    elif method == "intersection":
        iqr_mask = outlier_detector(col, "iqr", k=k, clip=False)
        z_mask = outlier_detector(col, "zscore", z_thresh=z_thresh, clip=False)
        mask = iqr_mask & z_mask

        if clip:
            q1, q3 = col.quantile(0.25), col.quantile(0.75)
            iqr_val = q3 - q1
            iqr_lower, iqr_upper = q1 - k * iqr_val, q3 + k * iqr_val
            result = col.copy()
            result[mask] = result[mask].clip(iqr_lower, iqr_upper)
            return result
        return mask

In [ ]:
for c in numeric_columns:
    print(c, int(outlier_detector(data[c]).sum()))

In [ ]:
cols_to_clip = ["points_per_game", "assists_per_game", "rebounds_per_game",
                "minutes_played", "true_shooting_pct"]

for c in cols_to_clip:
    data[f"{c} clip"] = outlier_detector(data[c], clip=True)

In [ ]:
fig, axes = plt.subplots(len(cols_to_clip), 2, figsize=(10, len(cols_to_clip) * 3))

for i, c in enumerate(cols_to_clip):
    sns.boxplot(y=data[c], ax=axes[i, 0], color="#DD8452")
    axes[i, 0].set_title(f"{c} - before")

    sns.boxplot(y=data[f"{c} clip"], ax=axes[i, 1], color="#DD8452")
    axes[i, 1].set_title(f"{c} - after")

fig.suptitle("Before & After Clipping Outliers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.subplots_adjust(top=0.96)
plt.show()

### Binning

Binning turns a continuous variable into a smaller number of meaningful groups — useful when the
raw scale is noisy but a coarser grouping tells a clearer story (e.g. career stage instead of
exact years of experience).

In [ ]:
def binning(data, col, method="custom", bins=5, thresholds=None, labels=None):

    df = data.copy()
    new_col = col + "_bin"

    if method == "equal width":
        df[new_col] = pd.cut(df[col], bins=bins, labels=labels)
    elif method == "quantile":
        df[new_col] = pd.qcut(df[col], q=bins, labels=labels, duplicates="drop")
    elif method == "custom":
        df[new_col] = pd.cut(df[col], bins=thresholds, labels=labels)

    print("column:", new_col)
    print("number of groups:", df[new_col].nunique())

    return df

In [ ]:
data = binning(data, "height_cm", method="custom",
               thresholds=[0, 190, 198, 206, 260],
               labels=["Short (<190cm)", "Average (190-198cm)", "Tall (198-206cm)", "Very Tall (206cm+)"])
data["height_cm_bin"].value_counts().sort_index()

In [ ]:
data = binning(data, "experience_years", method="custom",
               thresholds=[-0.01, 1, 4, 9, 30],
               labels=["Rookie", "Developing (2-4 yr)", "Prime (5-9 yr)", "Veteran (10+ yr)"])
data["experience_years_bin"].value_counts().sort_index()

In [ ]:
data = binning(data, "points_per_game", method="custom",
               thresholds=[-0.01, 5, 12, 20, 40],
               labels=["Bench (<5)", "Role Player (5-12)", "Starter (12-20)", "Star (20+)"])
data["points_per_game_bin"].value_counts().sort_index()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.countplot(data=data, x="height_cm_bin", color="#4C72B0", ax=axes[0])
axes[0].set_title("Height groups")
axes[0].tick_params(axis="x", rotation=30)

sns.countplot(data=data, x="experience_years_bin", color="#4C72B0", ax=axes[1])
axes[1].set_title("Career-stage groups")
axes[1].tick_params(axis="x", rotation=30)

sns.countplot(data=data, x="points_per_game_bin", color="#4C72B0", ax=axes[2])
axes[2].set_title("Scoring-tier groups")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

### Reclassifying categorical variables

In [ ]:
def reclassify_categorical(data, col, mapping_fn, new_col=None):
    data = data.copy()
    new_col = new_col or f"{col}_recls"
    data[new_col] = data[col].apply(mapping_fn)
    return data

In [ ]:
# US states (birth_place is a US state for domestic players, a country name otherwise)
US_STATES = {
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
    "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
    "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio",
    "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia",
    "Wisconsin", "Wyoming", "District of Columbia"
}

def origin_group(place):
    if pd.isna(place):
        return "Unknown"
    return "USA" if place in US_STATES else "International"

data = reclassify_categorical(data, "birth_place", origin_group, new_col="player_origin")
data["player_origin"].value_counts()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x="player_origin", color="#55A868")
plt.title("Player-seasons by birthplace origin")
plt.tight_layout()
plt.show()

### Final Columns

In [ ]:
data.tail()

## Column Descriptions

### Numeric Columns

- **minutes_played**: Total minutes played by the player during the season.
- **field_goals**: Number of successful field goals (2PT + 3PT combined) made.
- **field_goals_attempted**: Number of field goal attempts (2PT + 3PT combined).
- **three_point_field_goals**: Number of successful three-point shots made.
- **three_point_field_goals_attempted**: Number of three-point shot attempts.
- **two_point_field_goals**: Number of successful two-point shots made.
- **two_point_field_goals_attempted**: Number of two-point shot attempts.
- **effective_field_goal_percentage**: Shooting efficiency metric that adjusts for the extra value of 3-pointers: $$eFG\% = \frac{FG + 0.5 \times 3P}{FGA}$$
- **free_throws**: Number of successful free throws made.
- **free_throws_attempted**: Number of free throw attempts.
- **offensive_rebounds**: Number of rebounds collected on the offensive end.
- **defensive_rebounds**: Number of rebounds collected on the defensive end.
- **assists**: Number of passes leading directly to a made basket.
- **steals**: Number of times the player took the ball away from an opponent.
- **blocks**: Number of opponent shots blocked.
- **turnovers**: Number of times the player lost possession of the ball.
- **points**: Total points scored during the season.
- **personal_fouls**: Number of personal fouls committed.
- **triple_doubles**: Number of games where the player recorded double digits in three statistical categories.
- **rank**: Player's rank/ranking position in the dataset (e.g., by points or performance).
- **points_per_game**: Average points scored per game ($PPG = \frac{points}{games}$).
- **assists_per_game**: Average assists per game ($APG = \frac{assists}{games}$).
- **rebounds_per_game**: Average total rebounds per game ($RPG = \frac{total\_rebounds}{games}$).
- **fg_pct**: Field goal percentage ($FG\% = \frac{FG}{FGA}$).
- **ft_pct**: Free throw percentage ($FT\% = \frac{FT}{FTA}$).
- **true_shooting_pct**: Advanced shooting efficiency metric accounting for 2PT, 3PT, and free throws: $$TS\% = \frac{PTS}{2 \times (FGA + 0.44 \times FTA)}$$
- **height_cm**: Player's height in centimeters.
- **weight_kg**: Player's weight in kilograms.
- **games**: Number of games played during the season.
- **games_started**: Number of games the player started.
- **experience_years**: Number of years since the player's NBA debut, computed relative to the current season.
- **points_per_game clip**: Clipped version of `points_per_game`, used to reduce the effect of outliers.
- **assists_per_game clip**: Clipped version of `assists_per_game`.
- **rebounds_per_game clip**: Clipped version of `rebounds_per_game`.
- **minutes_played clip**: Clipped version of `minutes_played`.
- **true_shooting_pct clip**: Clipped version of `true_shooting_pct`.

### Categorical Columns

- **position_main**: Player's primary position (e.g., Guard, Forward, Center), extracted from the full `position` string.
- **team_name**: Name of the team the player played for during the season.
- **season**: NBA season identifier (e.g., "2020-21").
- **Target**: Did player win MVP?
- **shoots**: Player's shooting hand (Left/Right).
- **is_active**: Whether the player is currently active in the league.
- **player_origin**: Player's country or region of origin.
- **is_undrafted**: Whether the player entered the league without being drafted.
- **height_cm_bin**: Binned/categorized version of `height_cm` (e.g., "Short (<190cm)", "Tall (>200cm)").
- **experience_years_bin**: Binned/categorized version of `experience_years` (e.g., "Rookie", "Veteran").
- **points_per_game_bin**: Binned/categorized version of `points_per_game` (e.g., "Low scorer", "High scorer").

In [ ]:
numeric_columns_pp = ["minutes_played", "field_goals", "field_goals_attempted",
                   "three_point_field_goals", "three_point_field_goals_attempted",
                   "two_point_field_goals", "two_point_field_goals_attempted",
                   "effective_field_goal_percentage",
                   "free_throws", "free_throws_attempted",
                   "offensive_rebounds", "defensive_rebounds", "assists", "steals", "blocks",
                   "turnovers", "points", "personal_fouls", "triple_doubles", "rank",
                   "points_per_game", "assists_per_game", "rebounds_per_game",
                   "fg_pct", "ft_pct", "true_shooting_pct", "height_cm", "weight_kg",
                   "games", "games_started", "experience_years",

                      "points_per_game clip", "assists_per_game clip", "rebounds_per_game clip",
                      "minutes_played clip", "true_shooting_pct clip"]

categorical_columns_pp = ["position_main", "team_name", "season",
                        "Target", "shoots", "is_active", "player_origin",
                          "is_undrafted",
                          "height_cm_bin", "experience_years_bin", "points_per_game_bin"]

# Visualization

In [ ]:
Blue = "#7B9FF9"
Orange = "#F7A98B"
TARGET_PALETTE = {0: Orange, 1: Blue}

FALLBACK_PALETTE = ["#7B9FF9", "#BDAF88", "#87A16A", "#DAB552", "#C0A2D4", "#66BB87"]

sns.set_theme(style="whitegrid", font_scale=1.0)

import matplotlib.colors as mcolors
import matplotlib.cm as cm

base = cm.get_cmap("coolwarm")
colors = base(np.linspace(0.20, 0.80, 256))
soft_coolwarm = mcolors.LinearSegmentedColormap.from_list("soft_coolwarm", colors)

## Data overview after preprocessing

In [ ]:
def overview(df, col, k=1.5):

    # numeric
    if col in numeric_columns_pp:

        clean = df[col].replace([np.inf, -np.inf], np.nan).dropna()
        q1, q3 = clean.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - k * iqr, q3 + k * iqr
        n_lower = (clean < lower).sum()
        n_upper = (clean > upper).sum()

        stat, p_shapiro = stats.shapiro(clean.sample(min(len(clean), 5000), random_state=42))

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(col, fontsize=14, fontweight="bold")

        sns.boxplot(x=clean, ax=axes[0], color=Blue, fliersize=4)
        axes[0].set_title("Boxplot")
        axes[0].text(0.02, 0.9, f"lower: {n_lower}", transform=axes[0].transAxes)
        axes[0].text(0.02, 0.8, f"upper: {n_upper}", transform=axes[0].transAxes)

        sns.histplot(clean, kde=True, ax=axes[1], color=Blue, edgecolor="white")
        axes[1].set_title("Histogram + KDE")

        stats.probplot(clean, dist="norm", plot=axes[2])
        axes[2].get_lines()[1].set_color(Blue)
        axes[2].set_title(f"QQ-plot (p = {p_shapiro:.4f})")

        plt.tight_layout()
        plt.show()

    # categorical
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        fig.suptitle(col, fontsize=14, fontweight="bold")

        percent_table = pd.crosstab(df[col], columns="percent", normalize=True) * 100
        sns.heatmap(percent_table, annot=True, fmt=".1f", cmap=soft_coolwarm, cbar=False, ax=axes[0])
        axes[0].set_title("Share of rows (%)")

        order = df[col].value_counts().index
        sns.countplot(data=df, y=col, order=order, color="#4C72B0", ax=axes[1])
        axes[1].set_title("Count")

        plt.tight_layout()
        plt.show()

In [ ]:
for col in numeric_columns_pp[:10]:
    overview(data, col)

In [ ]:
for col in categorical_columns_pp:
    overview(data, col)

## Correlation Overview

#### For numeric columns

In [ ]:
def corr_heatmap(numeric_df, method="spearman"):
    cor = numeric_df.corr(method=method)
    plt.figure(figsize=(20,10))
    sns.heatmap(cor, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0,
                annot_kws={"size": 10})
    plt.title(f"Correlation Heatmap: {method}")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_heatmap(data[numeric_columns_pp], method="pearson")

In [ ]:
corr_heatmap(data[numeric_columns_pp], method="spearman")

## Two-variable comparison

### Numeric vs Target

In [ ]:
def numeric_vs_categorical_plot(df, col, target="Target"):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle(f"{col} by {target}", fontsize=14, fontweight="bold")

    target_values = sorted(df[target].dropna().unique())
    if set(target_values).issubset(TARGET_PALETTE.keys()):
        palette = TARGET_PALETTE
    else:
        palette = dict(zip(target_values, FALLBACK_PALETTE[:len(target_values)]))

    sns.histplot(data=df, x=col, hue=target, multiple="stack", palette=palette, ax=axes[0])
    axes[0].set_title("Stacked count")

    sns.histplot(data=df, x=col, hue=target, multiple="fill", palette=palette, ax=axes[1])
    axes[1].set_title("Proportion")

    sns.boxplot(data=df, x=target, y=col, palette=palette, ax=axes[2])
    axes[2].set_title("Boxplot")

    plt.tight_layout()
    plt.show()

In [ ]:
for col in numeric_columns_pp:
    numeric_vs_categorical_plot(data, col)

### Categorical vs Target

In [ ]:
def two_categorical_plot(df, col, target="Target"):
    fig, axes = plt.subplots(2, 2, figsize=(16, 9), squeeze=False)
    fig.suptitle(col, fontsize=14, fontweight="bold")

    order = df[col].value_counts().index
    target_values = sorted(df[target].dropna().unique())

    if set(target_values).issubset(TARGET_PALETTE.keys()):
        palette = TARGET_PALETTE
    else:
        palette = dict(zip(target_values, FALLBACK_PALETTE[:len(target_values)]))

    cont_table = pd.crosstab(df[col], df[target])
    sns.heatmap(cont_table, annot=True, fmt=".0f", cmap=soft_coolwarm, cbar=False, ax=axes[0, 0])

    prop_table = pd.crosstab(df[col], df[target], normalize="index")
    sns.heatmap(prop_table, annot=True, fmt=".2%", cmap=soft_coolwarm, cbar=False, ax=axes[0, 1])

    sns.countplot(data=df, x=col, hue=target, order=order, palette=palette, ax=axes[1, 0])
    axes[1, 0].tick_params(axis="x", rotation=45)
    axes[1, 0].set_title("Grouped count")

    ct = pd.crosstab(df[col], df[target], normalize="index").reindex(order)
    ct.plot(kind="bar", stacked=True, color=[palette[v] for v in ct.columns], ax=axes[1, 1])
    axes[1, 1].tick_params(axis="x", rotation=45)
    axes[1, 1].set_title("Proportion by " + target)

    plt.tight_layout()
    plt.show()

In [ ]:
two_categorical_plot(data, "position_main")

In [ ]:
two_categorical_plot(data, "height_cm_bin")

# Exploratory Data Analysis (EDA)

### Confidence Interval Analysis

- Confidence intervals estimate the likely range for the true population mean of a numeric
  variable, based on the sample we have.
- Applied here to `points_per_game`, `minutes_played`, and `true_shooting_pct` — instead of
  reporting only the sample mean, we report a 95% confidence interval so the result also
  communicates uncertainty.

In [ ]:
def mean_ci(data, column, confidence=0.95):

    x = data[column].replace([np.inf, -np.inf], np.nan).dropna()
    n = len(x)
    mean = x.mean()
    std = x.std(ddof=1)
    se = std / np.sqrt(n)

    alpha = 1 - confidence
    t_critical = stats.t.ppf(1 - alpha / 2, df=n - 1)
    margin_error = t_critical * se

    result = pd.DataFrame({
        "Variable": [column],
        "Type": "Mean CI",
        "N": n,
        "Sample Mean": round(mean, 3),
        "Sample Std": round(std, 3),
        "Confidence Level": confidence,
        "CI Lower": round(mean - margin_error, 3),
        "CI Upper": round(mean + margin_error, 3),
    })

    return result

In [ ]:
mean_ci(data, "points_per_game")

In [ ]:
mean_ci(data, "minutes_played")

In [ ]:
mean_ci(data, "true_shooting_pct")

### ANOVA: numeric outcome across a multi-category variable

In [ ]:
def anova_test(data, multi_cat_col, target_col):

    groups = []
    for group in data[multi_cat_col].dropna().unique():
        sample = data.loc[data[multi_cat_col] == group, target_col].replace([np.inf, -np.inf], np.nan).dropna()
        if len(sample) > 0:
            groups.append(sample)

    f_stat, p = stats.f_oneway(*groups)
    result = "Significant" if p < 0.05 else "Not Significant"

    return pd.DataFrame({"multi_category": [multi_cat_col], "target": [target_col],
                          "F_stat": f_stat, "P_value": p, "result": result})

In [ ]:
anova_test(data, "position_main", "points_per_game")

In [ ]:
anova_test(data, "height_cm_bin", "rebounds_per_game")

In [ ]:
anova_test(data, "experience_years_bin", "true_shooting_pct")

### Chi-square test of independence between two categorical variables

- **Null Hypothesis (H0):** the two categorical variables are independent.
- **Alternative Hypothesis (H1):** the two categorical variables are associated.
- If p-value < 0.05, reject H0.

In [ ]:
def compare_two_categorical(df, col_1, col_2, alpha=0.05):

    two_categorical_plot(df, col_1, col_2)

    contingency_table = pd.crosstab(df[col_1], df[col_2])
    chi2_stat, p_value, dof, expected_table = chi2_contingency(contingency_table)

    print(f"chi2_stat: {chi2_stat:.4f}")
    print(f"p-value: {p_value:.4g}")

    if p_value < alpha:
        print("Reject the Null Hypothesis")
        print(f"There IS a significant association between '{col_1}' and '{col_2}'.")
    else:
        print("Fail to Reject the Null Hypothesis")
        print(f"There is NO significant association between '{col_1}' and '{col_2}'.")

In [ ]:
compare_two_categorical(data, "position_main", "Target")

In [ ]:
compare_two_categorical(data, "height_cm_bin", "Target")

In [ ]:
compare_two_categorical(data, "experience_years_bin", "Target")

In [ ]:
compare_two_categorical(data, "is_undrafted", "Target")

### Numeric vs categorical (binary target)

To compare a continuous feature against a binary target, first check normality per group
(Shapiro-Wilk). If both groups are normal, use an independent-samples t-test. If not, attempt
a Box-Cox transform and re-test; if it still isn't normal, fall back to the non-parametric
Mann-Whitney U test.

In [ ]:
def numeric_vs_categorical(data, col_1, col_2, alpha=0.05):

    numeric_vs_categorical_plot(data, col_1, col_2)

    categories = data[col_2].dropna().unique()
    X1 = data.loc[data[col_2] == categories[0], col_1].replace([np.inf, -np.inf], np.nan).dropna()
    X2 = data.loc[data[col_2] == categories[1], col_1].replace([np.inf, -np.inf], np.nan).dropna()

    shapiro_p0 = stats.shapiro(X1.sample(min(len(X1), 5000), random_state=42)).pvalue
    shapiro_p1 = stats.shapiro(X2.sample(min(len(X2), 5000), random_state=42)).pvalue
    print(f"Shapiro group '{categories[0]}': p = {shapiro_p0:.4g}")
    print(f"Shapiro group '{categories[1]}': p = {shapiro_p1:.4g}")

    if shapiro_p0 > alpha and shapiro_p1 > alpha:
        t, p = stats.ttest_ind(X1, X2)
        print("Normal distribution -> t-test")
        print(f"t = {t:.4f}, p = {p:.4g}")
    else:
        try:
            X1_bc, _ = stats.boxcox(X1 - X1.min() + 1)
            X2_bc, _ = stats.boxcox(X2 - X2.min() + 1)
            shapiro_p0_bc = stats.shapiro(X1_bc.sample(min(len(X1_bc), 5000)) if hasattr(X1_bc, "sample") else X1_bc[:5000]).pvalue
            shapiro_p1_bc = stats.shapiro(X2_bc[:5000]).pvalue

            if shapiro_p0_bc > alpha and shapiro_p1_bc > alpha:
                t, p = stats.ttest_ind(X1_bc, X2_bc)
                print("Normal after Box-Cox -> t-test")
                print(f"t = {t:.4f}, p = {p:.4g}")
            else:
                u, p = stats.mannwhitneyu(X1, X2, alternative="two-sided")
                print("Not normal after Box-Cox -> Mann-Whitney U")
                print(f"U = {u:.1f}, p = {p:.4g}")
        except Exception:
            u, p = stats.mannwhitneyu(X1, X2, alternative="two-sided")
            print("Box-Cox not applicable -> Mann-Whitney U")
            print(f"U = {u:.1f}, p = {p:.4g}")

    result = "Significant" if p < alpha else "Not Significant"
    print("Result:", result)
    return p

In [ ]:
numeric_vs_categorical(data, "true_shooting_pct", "Target")

In [ ]:
numeric_vs_categorical(data, "points_per_game", "Target")

In [ ]:
numeric_vs_categorical(data, "minutes_played", "Target")

---
## Inter-Feature Relationships

The tests below examine relationships *between* independent features, not vs. the target.
This helps spot multicollinearity and see how player attributes cluster together.

In [ ]:
compare_two_categorical(data, "position_main", "height_cm_bin")

**Result to check after running:** position and height are expected to be strongly
associated by definition (centers are taller than guards, Section "Performance by Position"
in the earlier version of this notebook already confirmed this with ANOVA/Kruskal-Wallis) —
this chi-square test gives the categorical-vs-categorical version of the same fact.

In [ ]:
numeric_vs_categorical(data, "minutes_played", "is_undrafted")

**Result to check after running:** if undrafted players who make an NBA roster get
fewer minutes than drafted players, that would suggest teams give drafted players a longer
leash even at comparable performance — worth checking against `true_shooting_pct` too.

## KPIs

---
## Has Players' "Innate Ability" Improved? (Champion-Team Case Study)

**Claim:** an expert argues that, thanks to modern progress and better conditions, people's ability to develop and express their innate talent has improved compared to the past. As supporting evidence, the expert claims that the average *innate ability* of the players on the championship-winning team is higher in the **last 2 seasons** than it was **2 seasons before that**. The expert defines *innate ability* as **experience relative to age** (experience ÷ age) — the idea being that someone who has accumulated more career experience per year of life has converted more of their life into skill.

**Test plan:**
1. Find the champion team for each of the 4 most recent seasons that have a recorded champion (via the `champ` table).
2. Split them into two groups: the **last 2 seasons'** champions vs. the **2 seasons before those**.
3. For each champion team-season, take the players who were active on that roster that season (played at least one game), and compute each player's age and experience *as of that specific season* — not their current age/experience — since the claim is about that season's roster.
4. Compute `innate_ability = experience_at_season / age_at_season` per player and compare the two groups with an appropriate statistical test.

**Note on the experience definition:** the `players` table only stores a player's *current* total `experience_years`, which would be historically inaccurate for a season that already happened (e.g. it would overstate experience in an older season). Instead, experience at a given season is reconstructed as `season_year - nba_debut_year`, which gives the correct, season-specific experience for any point in a player's career.

In [ ]:
# Step 1: identify the champion team for each of the 4 most recent seasons on record
def parse_start_year(season_label):
    return int(str(season_label)[:4])

seasons_ext = seasons.copy()
seasons_ext["season_year"] = seasons_ext["season"].apply(parse_start_year)

champ_recent = champ.merge(seasons_ext, on="season_id").sort_values("season_id", ascending=False).head(4).sort_values("season_id").reset_index(drop=True)

print("4 most recent seasons with a recorded champion:")
print(champ_recent[["season_id", "season", "season_year", "team_id"]].to_string(index=False))

# Most recent 2 seasons vs. the 2 seasons before those
last_2_seasons = champ_recent.iloc[2:]
prior_2_seasons = champ_recent.iloc[:2]

print("\nLast 2 seasons' champions:")
print(last_2_seasons[["season", "team_id"]].to_string(index=False))
print("\nPrior 2 seasons' champions:")
print(prior_2_seasons[["season", "team_id"]].to_string(index=False))

In [ ]:
# Step 2: the roster of "active that season" players (played at least one game)
# for a given champion team-season, with season-specific age and experience
def champion_roster(season_id, season_year, team_id):
    roster = players_stats.loc[
        (players_stats["season_id"] == season_id) &
        (players_stats["team_id"] == team_id) &
        (players_stats["games"] > 0)
    ].copy()

    roster = roster.merge(
        players[["player_id", "name", "birth_date", "nba_debut"]],
        on="player_id", how="left"
    )

    roster["birth_year"] = pd.to_datetime(roster["birth_date"], errors="coerce").dt.year
    roster["age_at_season"] = season_year - roster["birth_year"]

    # experience at that season = years since NBA debut, not the player's current total
    roster["experience_at_season"] = (season_year - roster["nba_debut"]).clip(lower=0)

    roster = roster[roster["age_at_season"] > 0]
    roster["innate_ability"] = roster["experience_at_season"] / roster["age_at_season"]

    roster["season"] = season_year
    roster["team_id"] = team_id
    return roster[["player_id", "name", "season", "team_id", "age_at_season",
                    "experience_at_season", "innate_ability"]]

def build_group(season_rows):
    rosters = [
        champion_roster(row.season_id, row.season_year, row.team_id)
        for row in season_rows.itertuples()
    ]
    return pd.concat(rosters, ignore_index=True)

group_recent = build_group(last_2_seasons)
group_prior = build_group(prior_2_seasons)

print(f"Players (season-rows) in the 'last 2 seasons' champion group: {len(group_recent)}")
print(f"Players (season-rows) in the 'prior 2 seasons' champion group: {len(group_prior)}")

group_recent.head()

In [ ]:
# Step 3: compare the two groups' innate-ability distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.boxplot(data=[group_prior["innate_ability"], group_recent["innate_ability"]], ax=axes[0], palette=[Blue, Orange])
axes[0].set_xticklabels(["Prior 2 seasons' champions", "Last 2 seasons' champions"])
axes[0].set_title("Innate ability (experience / age) by group")

sns.kdeplot(group_prior["innate_ability"], fill=True, label="Prior 2 seasons", color=Blue, ax=axes[1])
sns.kdeplot(group_recent["innate_ability"], fill=True, label="Last 2 seasons", color=Orange, ax=axes[1])
axes[1].set_title("Distribution comparison")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Mean innate ability, prior 2 seasons' champions: {group_prior['innate_ability'].mean():.4f}")
print(f"Mean innate ability, last 2 seasons' champions:  {group_recent['innate_ability'].mean():.4f}")

In [ ]:
# Step 4: statistical test -- check normality first, then pick t-test or Mann-Whitney U
alpha = 0.05

shapiro_p_recent = stats.shapiro(group_recent["innate_ability"]).pvalue
shapiro_p_prior = stats.shapiro(group_prior["innate_ability"]).pvalue
print(f"Shapiro (last 2 seasons' champions):  p = {shapiro_p_recent:.4g}")
print(f"Shapiro (prior 2 seasons' champions): p = {shapiro_p_prior:.4g}")

if shapiro_p_recent > alpha and shapiro_p_prior > alpha:
    stat, p = stats.ttest_ind(group_recent["innate_ability"], group_prior["innate_ability"],
                               alternative="greater")
    test_used = "t-test (one-sided: recent > prior)"
else:
    stat, p = stats.mannwhitneyu(group_recent["innate_ability"], group_prior["innate_ability"],
                                  alternative="greater")
    test_used = "Mann-Whitney U (one-sided: recent > prior)"

print(f"\nTest used: {test_used}")
print(f"statistic = {stat:.4f}, p = {p:.4g}")

p < alpha (0.9802 < 0.05):

Result: fail to reject H0 

- no significant evidence, in this specific example, that innate ability (experience/age) was higher on the recent champion roster 
than on the one from 2 seasons before.
This does not confirm the expert's general claim.

- This tests one specific example (one team's roster, twice), not the general claim about society or the league as a whole — even a clear result here would only support the claim's illustration, not prove the broader statement.
- A champion roster's experience mix is shaped by team-building strategy (a team can win by combining a few veteran stars with young role players), not just by how much "innate ability" individual players have converted to skill — the metric conflates roster construction with individual development.
- `experience_at_season` uses `nba_debut` as the start of a career; it does not account for time missed to injury or time spent outside the NBA (overseas, G League), so it is a reasonable but imperfect proxy for actual playing experience.
- With only two seasons per group, the sample size is small and roster-dependent — a different pair of seasons or a different champion could change the conclusion.